# QBE: sync the rules to the rules table

Writes the rules in the YAML files (`config/tables/<table>.yml` and the column files they list) to the rules table
set under `rules.table` in `config/validation.yml` (`qbe_dq_rules`), one row per rule. Run it after changing the YAML
files; `qbe_validation.ipynb` reads the table when `rules.source` is `table`.

The table is **replaced** on each run, so it always matches the YAML files: rules removed from the YAML are removed
from the table, and edits made directly in the table are overwritten. `${name}` placeholders (e.g.
`${source_catalog}`) are stored as written and resolved by `qbe_validation.ipynb` when it runs.

**Parameters:** the `variables` in `validation.yml` that the rules table's name uses (e.g. `results_catalog`).
Needs only PyYAML (no DQX).

In [ ]:
import sys
from pathlib import Path

import yaml
from pyspark.sql import functions as F

# Config paths are relative to use_cases/qbe, the parent of this notebook's folder (notebooks); its src folder
# holds the com.qbe.de package
SRC_DIR = Path.cwd().parent
sys.path.insert(0, str(SRC_DIR / "src"))
from com.qbe.de.rules import RULES_SCHEMA, read_yaml_tables, to_rule_rows

raw_settings = yaml.safe_load((SRC_DIR / "config" / "validation.yml").read_text())["validation"]
rules_table = raw_settings["rules"]["table"]
# Widgets only for the variables in the rules table's name (e.g. results_catalog)
variables = {}
for name, default in (raw_settings.get("variables") or {}).items():
    if f"${{{name}}}" not in rules_table:
        continue
    try:
        dbutils.widgets.text(name, str(default))
        variables[name] = dbutils.widgets.get(name)
    except NameError:               # no dbutils outside Databricks: use the default
        variables[name] = str(default)

for name, value in variables.items():
    rules_table = rules_table.replace(f"${{{name}}}", value)

rows = to_rule_rows(read_yaml_tables(SRC_DIR / "config" / "tables"))
rules = spark.createDataFrame(rows, RULES_SCHEMA).withColumn("synced_ts", F.current_timestamp())
rules.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(rules_table)

print(f"{len(rows)} rules from config/tables synced to {rules_table}")
display(spark.read.table(rules_table).select("table_name", "column_name", "rule_id", "check_function", "enabled"))